# EDA — conjunto PT-BR validado por humanos

Esta análise descreve a amostra externa de avaliação. Ela não é usada para treinar o baseline Bitext.

In [1]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
PATH = ROOT / 'data/processed/ptbr_intent_validation_final.csv'
assert PATH.exists(), 'Execute primeiro: python scripts/finalize_ptbr_intent_dataset.py ...'
df = pd.read_csv(PATH, dtype=str, keep_default_na=False)
df[['sample_id', 'validated_intent', 'validation_method', 'has_uncertainty_flag', 'evaluation_eligible']].head()

,sample_id,validated_intent,validation_method,has_uncertainty_flag,evaluation_eligible
0,b2w-1529868ee5f31554,submit_product_feedback,double_agreement,False,True
1,b2w-39129ef55d968125,delivery_time,human_adjudication,False,True
2,b2w-0174ceaa16c23d46,product_issue,human_adjudication,True,False
3,b2w-43fde04da346da56,submit_product_feedback,double_agreement,False,True
4,b2w-8d2745f4cc98148f,delivery_time,human_adjudication,False,True


In [2]:
pd.Series({
    'registros': len(df),
    'intencoes_observadas': df['validated_intent'].nunique(),
    'concordancia_dupla': (df['validation_method'] == 'double_agreement').sum(),
    'adjudicacao_humana': (df['validation_method'] == 'human_adjudication').sum(),
    'incertos': (df['has_uncertainty_flag'] == 'True').sum(),
    'elegiveis': (df['evaluation_eligible'] == 'True').sum(),
})

registros               500
intencoes_observadas     20
concordancia_dupla      375
adjudicacao_humana      125
incertos                 78
elegiveis               422
dtype: int64

In [3]:
eligible = df[df['evaluation_eligible'].eq('True')]
pd.DataFrame({
    'total': df['validated_intent'].value_counts(),
    'elegiveis': eligible['validated_intent'].value_counts(),
}).fillna(0).astype(int).sort_values('total', ascending=False)

,total,elegiveis
validated_intent,,
submit_product_feedback,295,292
product_issue,67,46
delivery_issue,25,17
delivery_time,24,23
wrong_item,13,5
missing_item,12,5
product_information,12,11
damaged_delivery,11,6
submit_feedback,11,5


## Limite de uso

O conjunto contém feedbacks reais em português e rótulos humanos. Ainda assim, sua distribuição é concentrada e a amostragem foi estratificada por nota; ela não representa a distribuição operacional de um e-commerce. O modelo TF-IDF treinado no Bitext não deve receber uma métrica neste conjunto sem uma abordagem linguística apropriada.